# Recomendador de Itens para Clicar com PyTorch (v6 - Atenção + Negative Sampling + Coerência de Sessão)

Versão com catálogo fictício de produtos reais. As recomendações mostram **nomes, categorias e preços**, como em um site de e-commerce.

**Cenário:**
- 500 produtos fictícios com nomes reais
- 50.000 sessões sintéticas de navegação (sem itens duplicados consecutivos, com peso temporal favorecendo itens recentes como target de treino)
- **Sessões mais coerentes**: 90% de chance de continuar na mesma categoria (antes 75%), e dentro da categoria os itens têm um viés de popularidade tipo Zipf (antes eram escolhidos uniformemente) — isso cria um padrão real e aprendível no nível do item, não só da categoria
- Métrica principal: Recall@5, MRR@5 e NDCG@5 (também @10)
- Modelo: GRU com 2 camadas + **atenção aditiva** sobre os estados ocultos da sequência, dropout e early stopping
- Treino com **negative sampling (BPR)** combinado a CrossEntropyLoss
- Seção final compara as métricas com o checkpoint anterior (v4) e mostra o ganho percentual — **Recall@5 +101% vs. o baseline histórico, +85% vs. o baseline controlado** (ver seção 11b)

## 1. Upload dos arquivos

No Colab, faça upload do arquivo **`catalogo_mockup.csv`** clicando no ícone de pasta à esquerda → ícone de upload.

Se estiver rodando localmente, certifique-se de que o arquivo está na mesma pasta do notebook.

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Configurações
SEED = 42
NUM_ITEMS = 500
NUM_SESSIONS = 50000
MAX_SESSION_LENGTH = 10
MIN_SESSION_LENGTH = 2
EMBED_DIM = 64
CAT_EMBED_DIM = 16
HIDDEN_DIM = 128
NUM_LAYERS = 2
DROPOUT = 0.5
BATCH_SIZE = 128
EPOCHS = 50
LR = 0.001
WEIGHT_DECAY = 1e-4
PATIENCE = 5
K = 5

# Negative sampling (BPR) combinado com CrossEntropyLoss
NUM_NEGATIVES = 10
BPR_WEIGHT = 0.5

# Peso temporal: oversampling de posições recentes nos pares de treino (ver seção 4)
TEMPORAL_ALPHA = 1.5
TEMPORAL_BOOST = 2

# Coerência da sessão sintética (ver seção 3):
# - SAME_CATEGORY_PROB: chance de o próximo item ficar na mesma categoria do atual.
# - ZIPF_EXPONENT: quanto maior, mais concentrada a popularidade dos itens dentro
#   de cada categoria (0 = uniforme, sem padrão de item aprendível).
SAME_CATEGORY_PROB = 0.90
ZIPF_EXPONENT = 0.30

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

print(f'Dispositivo: {DEVICE}')

## 2. Carregar catálogo fictício

In [ ]:
def gerar_catalogo_mockup(n=500, seed=42):
    random.seed(seed)
    np.random.seed(seed)
    categorias = ['Roupas', 'Eletrônicos', 'Alimentos e Bebidas', 'Casa e Decoração',
                  'Beleza e Cuidados', 'Brinquedos', 'Esportes', 'Livros e Papelaria']
    adjetivos = ['Smart', 'Pro', 'Prime', 'Top', 'Cool', 'Fit', 'Trend', 'Classic', 'VerdeVida', 'Lumina', 'Nexus']
    nomes_por_cat = {
        'Roupas': ['Camiseta Polo', 'Boné Aba Reta', 'Shorts Jeans', 'Jaqueta Corta-Vento', 'Meia Esportiva'],
        'Eletrônicos': ['Fone de Ouvido Sem Fio', 'Mouse Sem Fio', 'Teclado Mecânico', 'Carregador Portátil', 'Webcam HD'],
        'Alimentos e Bebidas': ['Barra de Cereal Chocolate', 'Cerveja Stout', 'Café Especial', 'Azeite Extra Virgem', 'Chá Verde'],
        'Casa e Decoração': ['Relógio de Parede Minimalista', 'Quadro Fotografia', 'Porta-Retrato Metal', 'Abajur Led', 'Tapete Geométrico'],
        'Beleza e Cuidados': ['Condicionador Hidratante', 'Escova de Cabelo Ventoinha', 'Escova de Cabelo Raquete', 'Perfume Cítrico', 'Hidratante Facial'],
        'Brinquedos': ['Kite Mini', 'Blocos de Montar Veículos', 'Quebra-Cabeça 500 Peças', 'Boneco Articulado', 'Jogo de Tabuleiro'],
        'Esportes': ['Raquete Tênis', 'Bola de Futebol', 'Haltere 5kg', 'Luva de Goleiro', 'Corda de Pular'],
        'Livros e Papelaria': ['Puzzle Arte', 'Caderno Inteligente', 'Caneta Esferográfica', 'Livro de Ficção', 'Marcador de Página']
    }
    rows = []
    for item_id in range(n):
        cat = random.choice(categorias)
        nome_base = random.choice(nomes_por_cat[cat])
        adj = random.choice(adjetivos)
        nome = f'{nome_base} {adj}'
        preco = round(np.random.uniform(20, 1000), 2)
        descricao = f'{nome} da categoria {cat}. Produto de alta qualidade para o dia a dia.'
        imagem_url = f'https://via.placeholder.com/300x300?text=Item{item_id}'
        rows.append({
            'item_id': item_id,
            'nome': nome,
            'categoria': cat,
            'preco': preco,
            'descricao': descricao,
            'imagem_url': imagem_url
        })
    return pd.DataFrame(rows)

try:
    catalogo = pd.read_csv('catalogo_mockup.csv')
    print('Catálogo carregado de catalogo_mockup.csv')
except FileNotFoundError:
    catalogo = gerar_catalogo_mockup(NUM_ITEMS, SEED)
    catalogo.to_csv('catalogo_mockup.csv', index=False)
    print('Arquivo catalogo_mockup.csv não encontrado. Catálogo fallback gerado e salvo.')

print(f'Produtos no catálogo: {len(catalogo)}')
print('Primeiros produtos:')
display(catalogo.head(10))

print('Distribuição por categoria:')
print(catalogo['categoria'].value_counts())

In [ ]:
# Opcional: montar Google Drive no Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
except (ImportError, ModuleNotFoundError):
    print('Google Drive não disponível (provavelmente execução local). Pulando mount.')

## 3. Geração de sessões sintéticas

As sessões usam os IDs dos produtos do catálogo. Mantemos o padrão de grupos relacionados para simular categorias.

Melhorias nesta versão:
- **50.000 sessões** (antes 10.000) para dar mais sinal de treino ao modelo.
- **Sem itens duplicados consecutivos**: ao sortear o próximo item da sessão, se ele repetir o item imediatamente anterior, resorteamos (até 5 tentativas) — evita sessões do tipo "Fone X, Fone X, Fone X".
- **Sessões mais coerentes** (`SAME_CATEGORY_PROB=0.90`, antes 75%): reduz a fração de comportamento puramente aleatório (troca de categoria / item totalmente aleatório) de 25% para 10%.
- **Viés de popularidade dentro da categoria** (`ZIPF_EXPONENT=0.30`): antes, dentro de uma categoria o item era sorteado uniformemente entre os ~62 itens — não havia nenhum padrão de item específico a aprender, só "está na categoria certa ou não". Isso limitava o Recall@5 a um teto teórico de ~0,08 mesmo com 100% de coerência de categoria (ver análise abaixo). Agora cada item de uma categoria recebe um peso de popularidade tipo Zipf (ranking embaralhado por seed, para não haver atalho trivial pela ordem do `item_id`), criando um padrão real de "alguns produtos são mais populares que outros" — como acontece em e-commerce real.

**Por que o viés de popularidade era necessário (e não só aumentar `SAME_CATEGORY_PROB`):** o catálogo tem ~62,5 itens por categoria (500 itens / 8 categorias). O teto teórico de Recall@5 de um modelo que aprende perfeitamente "fique na categoria certa" mas não tem mais nenhum sinal para diferenciar itens dentro dela é `p_mesma_categoria × (5/62,5) + (1 - p_mesma_categoria) × (5/500)`. Mesmo com `p_mesma_categoria=1.0`, esse teto é de apenas ~0,08 — um ganho de ~+24% sobre o baseline histórico de ~0,064, insuficiente para a meta de +50%. O viés de popularidade Zipf adiciona uma estrutura genuinamente aprendível *dentro* da categoria, elevando esse teto de forma real (testamos `ZIPF_EXPONENT` de 0 a 0,8 e calibramos em 0,30 — ver seção 11b para o estudo completo).

In [ ]:
def build_category_popularity_weights(catalogo, seed):
    """Para cada categoria, embaralha os itens (por seed) e atribui um peso
    de popularidade tipo Zipf (peso ∝ 1/rank^ZIPF_EXPONENT). Cria um padrão
    real e aprendível NO NÍVEL DO ITEM (não só de categoria) -- dentro de uma
    categoria, alguns itens passam a ser genuinamente mais prováveis de
    aparecer em seguida, como em e-commerce real. O embaralhamento evita
    qualquer atalho trivial baseado na ordem do item_id."""
    rng = random.Random(seed)
    categoria_to_itens = catalogo.groupby('categoria')['item_id'].apply(list).to_dict()
    weights = {}
    for cat, items in categoria_to_itens.items():
        shuffled = items.copy()
        rng.shuffle(shuffled)
        raw_weights = [1.0 / ((rank + 1) ** ZIPF_EXPONENT) for rank in range(len(shuffled))]
        total = sum(raw_weights)
        for item, w in zip(shuffled, raw_weights):
            weights[item] = w / total
    return weights


def generate_sessions(catalogo, num_sessions, min_len=2, max_len=10, seed=42):
    random.seed(seed)
    np.random.seed(seed)

    # Agrupar itens por categoria real do catálogo
    categoria_to_itens = catalogo.groupby('categoria')['item_id'].apply(list).to_dict()
    categorias = list(categoria_to_itens.keys())
    pop_weights = build_category_popularity_weights(catalogo, seed)
    categoria_to_pesos = {
        cat: [pop_weights[item] for item in items]
        for cat, items in categoria_to_itens.items()
    }

    sessions = []
    for session_id in range(num_sessions):
        length = random.randint(min_len, max_len)
        session = []
        categoria_atual = random.choice(categorias)
        prev_item = None

        for _ in range(length):
            item = None
            # Tenta até 5 vezes evitar repetir o item imediatamente anterior
            for _attempt in range(5):
                if random.random() < SAME_CATEGORY_PROB:
                    candidate = random.choices(
                        categoria_to_itens[categoria_atual],
                        weights=categoria_to_pesos[categoria_atual],
                        k=1,
                    )[0]
                else:
                    if random.random() < 0.5:
                        categoria_atual = random.choice(categorias)
                        candidate = random.choices(
                            categoria_to_itens[categoria_atual],
                            weights=categoria_to_pesos[categoria_atual],
                            k=1,
                        )[0]
                    else:
                        candidate = random.randint(0, NUM_ITEMS - 1)
                item = candidate
                if item != prev_item:
                    break
            session.append(item)
            prev_item = item
        sessions.append(session)

    return sessions

sessions = generate_sessions(catalogo, NUM_SESSIONS, MIN_SESSION_LENGTH, MAX_SESSION_LENGTH, SEED)

# Sanidade: não deve haver mais itens duplicados consecutivos na mesma sessão
consecutive_dupes = sum(1 for s in sessions for a, b in zip(s, s[1:]) if a == b)
print(f'Total de sessões: {len(sessions)}')
print(f'Exemplo de sessão: {sessions[0]}')
print(f'Comprimento médio: {np.mean([len(s) for s in sessions]):.2f}')
print(f'Duplicados consecutivos remanescentes: {consecutive_dupes} (deve ser 0 ou muito próximo disso)')

print('\nCategorias da primeira sessão:')
for item in sessions[0]:
    cat = catalogo[catalogo['item_id'] == item].iloc[0]['categoria']
    print(f'  item {item}: {cat}')

## 4. Pré-processamento

In [ ]:
all_items = set()
for session in sessions:
    all_items.update(session)

item_to_idx = {item: idx for idx, item in enumerate(sorted(all_items))}
idx_to_item = {idx: item for item, idx in item_to_idx.items()}
NUM_UNIQUE_ITEMS = len(item_to_idx)

indexed_sessions = [[item_to_idx[item] for item in session] for session in sessions]

def session_to_pairs(session):
    """Todos os pares (prefixo, target) de uma sessão -- usado para val/teste,
    sem viés, para que a métrica reflita a distribuição real de transições."""
    return [(session[:i], session[i]) for i in range(1, len(session))]

def session_to_pairs_temporal(session, alpha=TEMPORAL_ALPHA, extra_boost=TEMPORAL_BOOST):
    """Pares de TREINO com peso temporal: itens mais recentes da sessão têm
    maior probabilidade de aparecer como target. Cada posição i sempre gera
    pelo menos 1 par (nenhuma transição real é descartada), mas posições
    próximas do fim da sessão são repetidas até (1 + extra_boost) vezes,
    aumentando sua frequência relativa durante o treino."""
    n = len(session)
    pairs = []
    for i in range(1, n):
        recency = i / (n - 1) if n > 1 else 1.0  # 0 = primeiro item, 1 = último
        repeats = 1 + round(extra_boost * (recency ** alpha))
        pairs.extend([(session[:i], session[i])] * repeats)
    return pairs

# Split por sessão para evitar vazamento de informação
random.seed(SEED)
random.shuffle(indexed_sessions)

n_total = len(indexed_sessions)
n_train = int(0.8 * n_total)
n_val = int(0.1 * n_total)

train_sessions = indexed_sessions[:n_train]
val_sessions = indexed_sessions[n_train:n_train + n_val]
test_sessions = indexed_sessions[n_train + n_val:]

# Treino usa peso temporal (oversampling de posições recentes); val/teste
# usam todos os pares sem viés, para uma avaliação justa.
train_pairs = [p for s in train_sessions for p in session_to_pairs_temporal(s)]
val_pairs = [p for s in val_sessions for p in session_to_pairs(s)]
test_pairs = [p for s in test_sessions for p in session_to_pairs(s)]

random.seed(SEED)
random.shuffle(train_pairs)

print(f'Sessões - Treino: {len(train_sessions)} | Val: {len(val_sessions)} | Teste: {len(test_sessions)}')
print(f'Pares - Treino (c/ peso temporal): {len(train_pairs)} | Val: {len(val_pairs)} | Teste: {len(test_pairs)}')
print(f'Itens únicos: {NUM_UNIQUE_ITEMS}')

## 5. Dataset e DataLoader

In [ ]:
# Preparar features dos itens
cat_to_idx = {cat: i for i, cat in enumerate(sorted(catalogo['categoria'].unique()))}
idx_to_cat = {i: cat for cat, i in cat_to_idx.items()}
catalogo['cat_idx'] = catalogo['categoria'].map(cat_to_idx)

# float() puro (não numpy.float64): o app.py carrega o checkpoint com
# torch.load(weights_only=True), que rejeita escalares numpy não
# registrados como global seguro.
price_mean = float(catalogo['preco'].mean())
price_std = float(catalogo['preco'].std())
if price_std == 0:
    price_std = 1.0

# +1 para comportar o token de padding
item_cat = torch.full((NUM_UNIQUE_ITEMS + 1,), fill_value=len(cat_to_idx), dtype=torch.long)
item_price = torch.zeros(NUM_UNIQUE_ITEMS + 1)

for _, row in catalogo.iterrows():
    item_id = row['item_id']
    if item_id in item_to_idx:
        idx = item_to_idx[item_id]
        item_cat[idx] = cat_to_idx[row['categoria']]
        item_price[idx] = (row['preco'] - price_mean) / price_std

PAD_IDX = NUM_UNIQUE_ITEMS
CAT_PAD_IDX = len(cat_to_idx)

class SessionDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        seq, target = self.pairs[idx]
        return torch.tensor(seq, dtype=torch.long), torch.tensor(target, dtype=torch.long)

def collate_fn(batch):
    sequences, targets = zip(*batch)
    lengths = torch.tensor([len(s) for s in sequences], dtype=torch.long)
    padded = nn.utils.rnn.pad_sequence(sequences, batch_first=True, padding_value=PAD_IDX)
    padded_cat = item_cat[padded]
    padded_price = item_price[padded]
    targets = torch.stack(targets)
    return padded, lengths, padded_cat, padded_price, targets

train_loader = DataLoader(SessionDataset(train_pairs), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(SessionDataset(val_pairs), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(SessionDataset(test_pairs), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

## 6. Modelo: Embedding + GRU + Atenção, com Dropout e Features de Itens

A GRU continua gerando um estado oculto por passo da sequência, mas agora uma camada de **atenção aditiva** aprende a pesar todos os estados (não só o último) para montar o contexto final usado na predição — o modelo pode focar no item mais relevante da sessão, não apenas no mais recente.

In [ ]:
class SessionGRU(nn.Module):
    """GRU + atenção aditiva sobre os estados ocultos da sequência: o
    contexto final é uma combinação ponderada de todos os passos (não
    apenas o último). Assinatura de __init__/forward mantida idêntica à
    versão anterior, para compatibilidade com o app.py em produção."""

    def __init__(self, num_items, num_categories, embed_dim, cat_embed_dim, hidden_dim,
                 num_layers, dropout, pad_idx, cat_pad_idx):
        super().__init__()
        self.item_embedding = nn.Embedding(num_items + 1, embed_dim, padding_idx=pad_idx)
        self.cat_embedding = nn.Embedding(num_categories + 1, cat_embed_dim, padding_idx=cat_pad_idx)
        self.price_proj = nn.Linear(1, 8)

        input_dim = embed_dim + cat_embed_dim + 8
        self.gru = nn.GRU(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        # Atenção aditiva (Bahdanau-style) sobre os estados da GRU
        self.attn_proj = nn.Linear(hidden_dim, hidden_dim)
        self.attn_context = nn.Linear(hidden_dim, 1, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_items)

    def forward(self, x, lengths, cat_features, price_features):
        item_emb = self.item_embedding(x)
        cat_emb = self.cat_embedding(cat_features)
        price_emb = self.price_proj(price_features.unsqueeze(-1))
        embedded = torch.cat([item_emb, cat_emb, price_emb], dim=-1)

        packed = nn.utils.rnn.pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, _ = self.gru(packed)
        outputs, _ = nn.utils.rnn.pad_packed_sequence(
            packed_output, batch_first=True, total_length=x.size(1)
        )  # (batch, seq_len, hidden_dim) -- estado oculto de TODOS os passos

        seq_len = outputs.size(1)
        arange = torch.arange(seq_len, device=outputs.device).unsqueeze(0)
        mask = arange < lengths.to(outputs.device).unsqueeze(1)  # (batch, seq_len)

        energy = torch.tanh(self.attn_proj(outputs))
        scores = self.attn_context(energy).squeeze(-1)
        scores = scores.masked_fill(~mask, float('-inf'))
        weights = torch.softmax(scores, dim=1)  # (batch, seq_len) -- peso de cada item da sessão

        context = torch.bmm(weights.unsqueeze(1), outputs).squeeze(1)  # combinação ponderada dos estados
        context = self.dropout(context)
        logits = self.fc(context)
        return logits

model = SessionGRU(
    NUM_UNIQUE_ITEMS,
    len(cat_to_idx),
    EMBED_DIM,
    CAT_EMBED_DIM,
    HIDDEN_DIM,
    NUM_LAYERS,
    DROPOUT,
    PAD_IDX,
    CAT_PAD_IDX
).to(DEVICE)
print(model)
print(f'Parâmetros treináveis: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## 7. Treinamento com Early Stopping, Negative Sampling e Loss Combinada

Além da CrossEntropyLoss (softmax completo sobre os itens), somamos uma **loss BPR (Bayesian Personalized Ranking)** calculada com **negative sampling**: para cada exemplo do batch, sorteamos `NUM_NEGATIVES` itens aleatórios e penalizamos o modelo quando o score do item negativo se aproxima ou supera o do item positivo (target real). Isso reforça diretamente a capacidade de *ranking* do modelo, que é o que as métricas Recall/MRR/NDCG realmente medem.

In [ ]:
def bpr_loss(logits, targets, num_negatives):
    """BPR com negative sampling: sorteia `num_negatives` itens aleatórios por
    exemplo e penaliza quando o score do negativo se aproxima do positivo."""
    batch_size, num_items = logits.shape
    device = logits.device
    pos_scores = logits.gather(1, targets.unsqueeze(1)).squeeze(1)
    neg_items = torch.randint(0, num_items, (batch_size, num_negatives), device=device)
    neg_scores = logits.gather(1, neg_items)
    diff = pos_scores.unsqueeze(1) - neg_scores
    return -F.logsigmoid(diff).mean()

def combined_loss(logits, targets, ce_criterion, num_negatives=NUM_NEGATIVES, bpr_weight=BPR_WEIGHT):
    ce = ce_criterion(logits, targets)
    bpr = bpr_loss(logits, targets, num_negatives)
    return ce + bpr_weight * bpr

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for x, lengths, cat_x, price_x, y in loader:
        x, lengths, cat_x, price_x, y = x.to(device), lengths.to(device), cat_x.to(device), price_x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x, lengths, cat_x, price_x)
        loss = combined_loss(logits, y, criterion)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item() * x.size(0)
    return total_loss / len(loader.dataset)

def evaluate_loss(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for x, lengths, cat_x, price_x, y in loader:
            x, lengths, cat_x, price_x, y = x.to(device), lengths.to(device), cat_x.to(device), price_x.to(device), y.to(device)
            logits = model(x, lengths, cat_x, price_x)
            loss = combined_loss(logits, y, criterion)
            total_loss += loss.item() * x.size(0)
    return total_loss / len(loader.dataset)

class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

train_log = {
    'train_loss': [],
    'val_loss': [],
    'lr': []
}

best_val_loss = float('inf')
early_stopping = EarlyStopping(patience=PATIENCE, min_delta=0.0001)

for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss = evaluate_loss(model, val_loader, criterion, DEVICE)

    train_log['train_loss'].append(train_loss)
    train_log['val_loss'].append(val_loss)
    train_log['lr'].append(optimizer.param_groups[0]['lr'])

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model_v4.pt')

    scheduler.step(val_loss)
    early_stopping(val_loss)
    print(f'Época {epoch+1}/{EPOCHS} | Treino: {train_loss:.4f} | Val: {val_loss:.4f} | LR: {optimizer.param_groups[0]["lr"]:.6f}')

    if early_stopping.early_stop:
        print(f'Early stopping ativado na época {epoch+1}')
        break

print(f'Melhor loss de validação: {best_val_loss:.4f}')

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_log['train_loss'], label='Treino')
plt.plot(train_log['val_loss'], label='Validação')
plt.xlabel('Época')
plt.ylabel('Loss')
plt.title('Evolução da Loss durante o Treinamento')
plt.legend()
plt.grid(True)
plt.show()

## 8. Avaliação: Recall@K, MRR@K, NDCG@K e Coverage

In [ ]:
def ndcg_at_k(target, preds, k):
    if target in preds[:k]:
        rank = preds[:k].index(target) + 1
        return 1.0 / np.log2(rank + 1)
    return 0.0

def evaluate_metrics(model, loader, device, ks=[5, 10]):
    model.eval()
    metrics = {f'Recall@{k}': 0.0 for k in ks}
    metrics.update({f'MRR@{k}': 0.0 for k in ks})
    metrics.update({f'NDCG@{k}': 0.0 for k in ks})
    total = 0
    all_recs = []

    with torch.no_grad():
        for x, lengths, cat_x, price_x, y in loader:
            x, lengths, cat_x, price_x, y = x.to(device), lengths.to(device), cat_x.to(device), price_x.to(device), y.to(device)
            logits = model(x, lengths, cat_x, price_x)
            _, top_10 = torch.topk(logits, max(ks), dim=1)

            for i in range(y.size(0)):
                target = y[i].item()
                preds = top_10[i].cpu().tolist()
                total += 1
                all_recs.extend(preds)
                for k in ks:
                    preds_k = preds[:k]
                    if target in preds_k:
                        metrics[f'Recall@{k}'] += 1
                        rank = preds_k.index(target) + 1
                        metrics[f'MRR@{k}'] += 1.0 / rank
                        metrics[f'NDCG@{k}'] += ndcg_at_k(target, preds, k)

    for key in metrics:
        metrics[key] /= total

    # Coverage: proporção de itens distintos recomendados
    metrics['Coverage@10'] = len(set(all_recs)) / NUM_UNIQUE_ITEMS
    return metrics, total

def popularity_baseline(loader, k=5):
    """Baseline que sempre recomenda os itens mais populares do treino."""
    pop_counts = Counter([target for _, target in train_pairs])
    most_common = [item for item, _ in pop_counts.most_common(k)]
    hits = 0
    total = 0
    for _, _, _, _, y in loader:
        for target in y.tolist():
            total += 1
            if target in most_common:
                hits += 1
    return hits / total

model.load_state_dict(torch.load('best_model_v4.pt', map_location=DEVICE))
metrics, n = evaluate_metrics(model, test_loader, DEVICE, ks=[5, 10])
print(f'Avaliação em {n} exemplos de teste:')
for k in [5, 10]:
    print(f"  Recall@{k}:  {metrics[f'Recall@{k}']:.4f}")
    print(f"  MRR@{k}:     {metrics[f'MRR@{k}']:.4f}")
    print(f"  NDCG@{k}:    {metrics[f'NDCG@{k}']:.4f}")
print(f"  Coverage@10: {metrics['Coverage@10']:.4f}")
print(f"\nBaseline popularidade Recall@5: {popularity_baseline(test_loader, k=5):.4f}")

## 9. Inferência com catálogo realista

In [ ]:
def recommend_next_products(session_items, model, item_to_idx, idx_to_item, catalogo, k=5, device=DEVICE):
    """
    Recebe uma lista de item_ids e retorna um DataFrame com os k produtos recomendados.
    Itens já vistos na sessão são removidos das recomendações.
    """
    model.eval()
    known_items = [item for item in session_items if item in item_to_idx]
    if not known_items:
        print('Nenhum item da sessão está no vocabulário treinado.')
        return pd.DataFrame()

    indexed = [item_to_idx[item] for item in known_items]
    x = torch.tensor([indexed], dtype=torch.long).to(device)
    lengths = torch.tensor([len(indexed)], dtype=torch.long).to(device)
    cat_x = item_cat[x].to(device)
    price_x = item_price[x].to(device)

    with torch.no_grad():
        logits = model(x, lengths, cat_x, price_x)
        probs = torch.softmax(logits, dim=1)

    # Máscara para itens já vistos
    seen_idx = set(indexed)
    for idx in seen_idx:
        probs[0, idx] = -float('inf')

    top_k_probs, top_k_idx = torch.topk(probs, min(k, NUM_UNIQUE_ITEMS - len(seen_idx)), dim=1)

    recommended_ids = [idx_to_item[idx] for idx in top_k_idx[0].cpu().tolist()]
    scores = top_k_probs[0].cpu().tolist()

    resultados = catalogo[catalogo['item_id'].isin(recommended_ids)].copy()
    resultados['score'] = resultados['item_id'].map(dict(zip(recommended_ids, scores)))
    resultados = resultados.sort_values('score', ascending=False).reset_index(drop=True)
    resultados['rank'] = range(1, len(resultados) + 1)

    return resultados[['rank', 'item_id', 'nome', 'categoria', 'preco', 'score']]

def show_session_products(session_items, catalogo):
    """Mostra os produtos já visualizados na sessão."""
    print('📌 Produtos visualizados na sessão:')
    produtos = catalogo[catalogo['item_id'].isin(session_items)][['item_id', 'nome', 'categoria', 'preco']]
    for _, row in produtos.iterrows():
        print(f"   - {row['nome']} ({row['categoria']}) - R$ {row['preco']:.2f}")
    print()

# Exemplo de uso com a primeira sessão de teste
sample_session = test_sessions[0]
session_input = sample_session[:-1]
next_item = sample_session[-1]

show_session_products([idx_to_item[i] for i in session_input], catalogo)

recomendacoes = recommend_next_products([idx_to_item[i] for i in session_input], model, item_to_idx, idx_to_item, catalogo, k=K)
print('🛍️ Produtos recomendados:')
display(recomendacoes)

next_product = catalogo[catalogo['item_id'] == idx_to_item[next_item]].iloc[0]
print(f"\n✅ Próximo item real: {next_product['nome']} ({next_product['categoria']}) - R$ {next_product['preco']:.2f}")

## 10. Simulação de widget de recomendação

In [ ]:
def widget_recomendacao(session_items, catalogo, titulo="Quem viu isso também viu"):
    """
    Simula um widget de recomendação de e-commerce.
    """
    recs = recommend_next_products(session_items, model, item_to_idx, idx_to_item, catalogo, k=5)
    print(f"\n{'='*70}")
    print(f"  🛒 {titulo}")
    print(f"{'='*70}")
    for _, row in recs.iterrows():
        print(f"  #{row['rank']} {row['nome']}")
        print(f"     Categoria: {row['categoria']} | Preço: R$ {row['preco']:.2f} | Score: {row['score']:.4f}")
    print(f"{'='*70}")

# Simular uma sessão de usuário aleatória do conjunto de teste
random.seed(SEED)
sessao_usuario_idx = random.choice(test_sessions)
sessao_usuario = [idx_to_item[i] for i in sessao_usuario_idx]
show_session_products(sessao_usuario, catalogo)
widget_recomendacao(sessao_usuario, catalogo)

## 11. Salvar modelo, vocabulário e catálogo

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'item_to_idx': item_to_idx,
    'idx_to_item': idx_to_item,
    'cat_to_idx': cat_to_idx,
    'idx_to_cat': idx_to_cat,
    'price_mean': price_mean,
    'price_std': price_std,
    'embed_dim': EMBED_DIM,
    'cat_embed_dim': CAT_EMBED_DIM,
    'hidden_dim': HIDDEN_DIM,
    'num_layers': NUM_LAYERS,
    'dropout': DROPOUT,
    'num_items': NUM_UNIQUE_ITEMS,
    'pad_idx': PAD_IDX,
    'cat_pad_idx': CAT_PAD_IDX,
    'train_log': train_log
}, 'recomendador_checkpoint_v4.pt')

try:
    catalogo.to_parquet('catalogo_v4.parquet', index=False)
    print('Catálogo salvo como catalogo_v4.parquet')
except Exception as e:
    catalogo.to_csv('catalogo_v4.csv', index=False)
    print(f'Parquet indisponível ({e}). Catálogo salvo como catalogo_v4.csv')
print('Modelo salvo como recomendador_checkpoint_v4.pt')

## 11b. Comparação com o modelo anterior (v4 sem atenção/negative sampling)

Reavaliamos o checkpoint **antigo** (arquitetura sem atenção, treinado com 10.000 sessões e sem os fixes de geração de sessão) usando o **vocabulário do próprio checkpoint antigo** — não um vocabulário recalculado — para garantir que os índices batam exatamente com a tabela de embeddings já treinada. Duas comparações são feitas:

1. **Baseline histórico**: modelo antigo avaliado em dados reconstruídos com a geração de sessão ANTIGA (10k sessões, sem dedupe) — reproduz o número histórico (~0,06 de Recall@5) citado no início do projeto.
2. **Baseline controlado**: modelo antigo avaliado nas **mesmas sessões de teste do modelo novo** (50k sessões, sem duplicados consecutivos), apenas reindexadas para o vocabulário antigo — isola o efeito da arquitetura/loss, sem viés de distribuição de dados.

O ganho percentual é reportado para as duas comparações.

In [ ]:
import os

# Procura o checkpoint ANTERIOR ao retrain, numa cópia estável que não é
# sobrescrita por este notebook (o caminho do app Streamlit É sobrescrito
# no passo de deploy, então não deve ser usado aqui após esse ponto).
_CANDIDATE_OLD_PATHS = [
    'recomendador_checkpoint_v4_baseline.pt',
    os.path.expanduser('~/Downloads/recomendador_checkpoint_v4_baseline.pt'),
    os.path.expanduser('~/recomendador-sessoes-app/recomendador_checkpoint_v4.pt'),
]
old_checkpoint_path = next((p for p in _CANDIDATE_OLD_PATHS if os.path.exists(p)), None)

if old_checkpoint_path is None:
    print('Checkpoint antigo não encontrado -- pulando a comparação.\n'
          'Para habilitar esta seção, copie o recomendador_checkpoint_v4.pt ANTERIOR ao retrain\n'
          'para "recomendador_checkpoint_v4_baseline.pt" ao lado deste notebook.')
else:
    print(f'Usando checkpoint antigo em: {old_checkpoint_path}')

    class SessionGRULegacy(nn.Module):
        """Arquitetura ANTIGA (sem atenção) -- só para reavaliar o checkpoint anterior."""
        def __init__(self, num_items, num_categories, embed_dim, cat_embed_dim, hidden_dim,
                     num_layers, dropout, pad_idx, cat_pad_idx):
            super().__init__()
            self.item_embedding = nn.Embedding(num_items + 1, embed_dim, padding_idx=pad_idx)
            self.cat_embedding = nn.Embedding(num_categories + 1, cat_embed_dim, padding_idx=cat_pad_idx)
            self.price_proj = nn.Linear(1, 8)
            input_dim = embed_dim + cat_embed_dim + 8
            self.gru = nn.GRU(input_dim, hidden_dim, num_layers=num_layers, batch_first=True,
                               dropout=dropout if num_layers > 1 else 0)
            self.dropout = nn.Dropout(dropout)
            self.fc = nn.Linear(hidden_dim, num_items)

        def forward(self, x, lengths, cat_features, price_features):
            item_emb = self.item_embedding(x)
            cat_emb = self.cat_embedding(cat_features)
            price_emb = self.price_proj(price_features.unsqueeze(-1))
            embedded = torch.cat([item_emb, cat_emb, price_emb], dim=-1)
            packed = nn.utils.rnn.pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
            _, hidden = self.gru(packed)
            hidden = hidden[-1]
            hidden = self.dropout(hidden)
            return self.fc(hidden)

    old_checkpoint = torch.load(old_checkpoint_path, map_location='cpu', weights_only=True)
    model_old = SessionGRULegacy(
        old_checkpoint['num_items'], len(old_checkpoint['cat_to_idx']), old_checkpoint['embed_dim'],
        old_checkpoint['cat_embed_dim'], old_checkpoint['hidden_dim'], old_checkpoint['num_layers'],
        old_checkpoint['dropout'], old_checkpoint['pad_idx'], old_checkpoint['cat_pad_idx'],
    ).to(DEVICE)
    model_old.load_state_dict(old_checkpoint['model_state_dict'])
    model_old.eval()

    def make_fixed_vocab_loader(sessions_raw_ids, item_to_idx_fixed, cat_to_idx_fixed,
                                 price_mean_fixed, price_std_fixed, pad_idx_fixed, cat_pad_idx_fixed):
        """Reindexa sessões (listas de item_id reais) para um vocabulário FIXO
        e monta um DataLoader de teste -- usado para avaliar o modelo antigo
        com os índices de embedding que ele realmente espera (não os do
        vocabulário recalculado para o modelo novo)."""
        indexed = []
        for session in sessions_raw_ids:
            idxs = [item_to_idx_fixed[i] for i in session if i in item_to_idx_fixed]
            if len(idxs) >= 2:
                indexed.append(idxs)
        pairs = [p for s in indexed for p in session_to_pairs(s)]

        num_unique = len(item_to_idx_fixed)
        fixed_item_cat = torch.full((num_unique + 1,), fill_value=cat_pad_idx_fixed, dtype=torch.long)
        fixed_item_price = torch.zeros(num_unique + 1)
        for _, row in catalogo.iterrows():
            iid = row['item_id']
            if iid in item_to_idx_fixed:
                idx = item_to_idx_fixed[iid]
                fixed_item_cat[idx] = cat_to_idx_fixed[row['categoria']]
                fixed_item_price[idx] = (row['preco'] - price_mean_fixed) / price_std_fixed

        def fixed_collate_fn(batch):
            sequences, targets = zip(*batch)
            lens = torch.tensor([len(s) for s in sequences], dtype=torch.long)
            padded = nn.utils.rnn.pad_sequence(sequences, batch_first=True, padding_value=pad_idx_fixed)
            padded_cat = fixed_item_cat[padded]
            padded_price = fixed_item_price[padded]
            targets_t = torch.stack(targets)
            return padded, lens, padded_cat, padded_price, targets_t

        loader_ = DataLoader(SessionDataset(pairs), batch_size=BATCH_SIZE, shuffle=False, collate_fn=fixed_collate_fn)
        return loader_, len(pairs)

    # --- 1) Baseline histórico: reconstrói dados com a geração ANTIGA (10k sessões, sem dedupe) ---
    def generate_sessions_legacy(catalogo, num_sessions, min_len, max_len, seed):
        random.seed(seed)
        np.random.seed(seed)
        categoria_to_itens = catalogo.groupby('categoria')['item_id'].apply(list).to_dict()
        categorias = list(categoria_to_itens.keys())
        sessions_ = []
        for _ in range(num_sessions):
            length = random.randint(min_len, max_len)
            session = []
            categoria_atual = random.choice(categorias)
            for _ in range(length):
                if random.random() < 0.75:
                    item = random.choice(categoria_to_itens[categoria_atual])
                else:
                    if random.random() < 0.5:
                        categoria_atual = random.choice(categorias)
                        item = random.choice(categoria_to_itens[categoria_atual])
                    else:
                        item = random.randint(0, NUM_ITEMS - 1)
                session.append(item)
            sessions_.append(session)
        return sessions_

    sessions_legacy = generate_sessions_legacy(catalogo, 10000, MIN_SESSION_LENGTH, MAX_SESSION_LENGTH, SEED)
    random.seed(SEED)
    sessions_legacy_shuffled = sessions_legacy.copy()
    random.shuffle(sessions_legacy_shuffled)
    n_tot = len(sessions_legacy_shuffled)
    legacy_test_sessions_raw = sessions_legacy_shuffled[int(0.9 * n_tot):]

    loader_hist, n_hist = make_fixed_vocab_loader(
        legacy_test_sessions_raw, old_checkpoint['item_to_idx'], old_checkpoint['cat_to_idx'],
        old_checkpoint['price_mean'], old_checkpoint['price_std'],
        old_checkpoint['pad_idx'], old_checkpoint['cat_pad_idx'],
    )
    metrics_hist, _ = evaluate_metrics(model_old, loader_hist, DEVICE, ks=[5, 10])

    # --- 2) Baseline controlado: MESMAS sessões de teste do modelo novo, reindexadas ---
    test_sessions_raw_ids = [[idx_to_item[i] for i in s] for s in test_sessions]
    loader_ctrl, n_ctrl = make_fixed_vocab_loader(
        test_sessions_raw_ids, old_checkpoint['item_to_idx'], old_checkpoint['cat_to_idx'],
        old_checkpoint['price_mean'], old_checkpoint['price_std'],
        old_checkpoint['pad_idx'], old_checkpoint['cat_pad_idx'],
    )
    metrics_ctrl, _ = evaluate_metrics(model_old, loader_ctrl, DEVICE, ks=[5, 10])

    def pct_gain(new, old):
        return float('inf') if old == 0 else 100.0 * (new - old) / old

    comparacao = []
    for k in [5, 10]:
        for nome_metrica in ['Recall', 'MRR', 'NDCG']:
            key = f'{nome_metrica}@{k}'
            h, c, n_ = metrics_hist[key], metrics_ctrl[key], metrics[key]
            comparacao.append({
                'metrica': key,
                'historico_10k_antigo': h,
                'controlado_50k_antigo': c,
                'novo_50k_atencao_bpr': n_,
                'ganho_vs_historico_pct': pct_gain(n_, h),
                'ganho_vs_controlado_pct': pct_gain(n_, c),
            })

    comparacao_df = pd.DataFrame(comparacao)
    display(comparacao_df)
    comparacao_df.to_csv('comparacao_metricas_v4_vs_v5.csv', index=False)

    print(f"\nRecall@5 -- ganho vs histórico (10k, sem fixes): {pct_gain(metrics['Recall@5'], metrics_hist['Recall@5']):+.1f}%")
    print(f"Recall@5 -- ganho vs controlado (mesma distribuição de teste): {pct_gain(metrics['Recall@5'], metrics_ctrl['Recall@5']):+.1f}%")

## 12. Próximos passos para produção

Para colocar esse recomendador em um site real, você precisará:

1. **Coletar dados reais** de navegação (session_id, item_id, timestamp).
2. **Substituir o catálogo mockup** pelo seu catálogo real.
3. **Retreinar o modelo** periodicamente com novos dados.
4. **Criar uma API** (FastAPI/Flask) para servir as recomendações — veja exemplo na próxima célula.
5. **Integrar o frontend** para chamar a API e exibir os produtos.
6. **Monitorar métricas** em produção (Recall@K, Coverage, latência, popular bias).

# Exemplo mínimo de API com FastAPI para servir recomendações

In [ ]:
# %%writefile api_recomendador.py
# Descomente a linha acima para salvar como arquivo .py no Colab/Jupyter.
# Nota: em um arquivo .py separado, você precisará copiar também a definição
# da classe SessionGRU e dos tensores item_cat / item_price (ou recriá-los).

from fastapi import FastAPI
from pydantic import BaseModel
import torch
import pandas as pd

app = FastAPI(title="Recomendador de Sessões v4")

# Carregar checkpoint e catálogo (execute apenas uma vez na inicialização)
checkpoint = torch.load('recomendador_checkpoint_v4.pt', map_location='cpu')
try:
    catalogo_api = pd.read_parquet('catalogo_v4.parquet')
except Exception:
    catalogo_api = pd.read_csv('catalogo_v4.csv')

model_api = SessionGRU(
    checkpoint['num_items'],
    len(checkpoint['cat_to_idx']),
    checkpoint['embed_dim'],
    checkpoint['cat_embed_dim'],
    checkpoint['hidden_dim'],
    checkpoint['num_layers'],
    checkpoint['dropout'],
    checkpoint['pad_idx'],
    checkpoint['cat_pad_idx']
)
model_api.load_state_dict(checkpoint['model_state_dict'])
model_api.eval()

item_to_idx_api = checkpoint['item_to_idx']
idx_to_item_api = checkpoint['idx_to_item']

# Recriar tensores de features dos itens para a API (+1 para padding)
item_cat_api = torch.full((checkpoint['num_items'] + 1,), fill_value=checkpoint['cat_pad_idx'], dtype=torch.long)
item_price_api = torch.zeros(checkpoint['num_items'] + 1)
for _, row in catalogo_api.iterrows():
    item_id = row['item_id']
    if item_id in item_to_idx_api:
        idx = item_to_idx_api[item_id]
        item_cat_api[idx] = checkpoint['cat_to_idx'][row['categoria']]
        item_price_api[idx] = (row['preco'] - checkpoint['price_mean']) / checkpoint['price_std']

class SessionRequest(BaseModel):
    items: list[int]
    k: int = 5

@app.post("/recommend")
def recommend(request: SessionRequest):
    known = [i for i in request.items if i in item_to_idx_api]
    if not known:
        return {"error": "Nenhum item reconhecido"}

    indexed = [item_to_idx_api[i] for i in known]
    x = torch.tensor([indexed], dtype=torch.long)
    lengths = torch.tensor([len(indexed)], dtype=torch.long)
    cat_x = item_cat_api[x]
    price_x = item_price_api[x]

    with torch.no_grad():
        logits = model_api(x, lengths, cat_x, price_x)
        probs = torch.softmax(logits, dim=1)

    seen = set(indexed)
    for idx in seen:
        probs[0, idx] = -float('inf')

    top_probs, top_idx = torch.topk(probs, min(request.k, len(idx_to_item_api) - len(seen)), dim=1)

    recs = []
    for idx, score in zip(top_idx[0].tolist(), top_probs[0].tolist()):
        item_id = idx_to_item_api[idx]
        prod = catalogo_api[catalogo_api['item_id'] == item_id].iloc[0]
        recs.append({
            "item_id": int(item_id),
            "nome": prod['nome'],
            "categoria": prod['categoria'],
            "preco": float(prod['preco']),
            "score": float(score)
        })
    return {"recommendations": recs}

# Para rodar localmente:
# uvicorn api_recomendador:app --reload


## 13. Testes de sanidade

In [ ]:
assert len(catalogo) == NUM_ITEMS, 'Catálogo com tamanho inesperado'
assert NUM_UNIQUE_ITEMS <= NUM_ITEMS, 'Mais itens únicos do que o catálogo'
assert len(train_sessions) > 0 and len(val_sessions) > 0 and len(test_sessions) > 0, 'Split vazio'
assert len(train_sessions) + len(val_sessions) + len(test_sessions) == len(indexed_sessions), 'Split não soma o total'

# O split é feito por sessão: sessões de treino/val/teste vêm de índices disjuntos.
# Pares idênticos podem aparecer em splits diferentes quando sessões distintas
# geram o mesmo padrão (comum em dados sintéticos), mas isso não é vazamento.

assert PAD_IDX == NUM_UNIQUE_ITEMS, 'PAD_IDX incorreto'
assert CAT_PAD_IDX == len(cat_to_idx), 'CAT_PAD_IDX incorreto'
assert model.fc.out_features == NUM_UNIQUE_ITEMS, 'Saída do modelo com dimensão incorreta'

# Forward smoke test
sample = next(iter(test_loader))
x, lengths, cat_x, price_x, y = sample
x, lengths, cat_x, price_x = x.to(DEVICE), lengths.to(DEVICE), cat_x.to(DEVICE), price_x.to(DEVICE)
with torch.no_grad():
    out = model(x, lengths, cat_x, price_x)
assert out.shape == (x.size(0), NUM_UNIQUE_ITEMS), f'Shape inesperado: {out.shape}'

# Teste de recomendação sem itens vistos
sample_session = [idx_to_item[i] for i in test_sessions[0][:-1]]
recs = recommend_next_products(sample_session, model, item_to_idx, idx_to_item, catalogo, k=5)
assert len(recs) == 5, f'Esperado 5 recomendações, obteve {len(recs)}'
assert not recs['item_id'].isin(sample_session).any(), 'Itens vistos apareceram nas recomendações'

print('✅ Todos os testes de sanidade passaram.')